In [ ]:
"""
Superstore Profitability Analysis - Exploratory Data Analysis
================================================================
Cleans the dataset and analyzes profitability drivers: category,
sub-category, discount, and geography.
"""

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# ------------------------------------------------------------------
# 1. Load & clean data
# ------------------------------------------------------------------
df = pd.read_csv("D:/Superstore/Data/SampleSuperstore.csv/SampleSuperstore.csv")

print("Shape before cleaning:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())

# Drop exact duplicate rows
dupes = df.duplicated().sum()
print(f"\nDuplicate rows found: {dupes}")
df = df.drop_duplicates()

# Postal Code is an identifier, not a quantity - treat as string
df["Postal Code"] = df["Postal Code"].astype(str)

print("Shape after cleaning:", df.shape)

# Save cleaned version
df.to_csv("D:/Superstore/Data/SampleSuperstore.csv/SampleSuperstore_cleaned.csv", index=False)

# ------------------------------------------------------------------
# 2. Overall profitability snapshot
# ------------------------------------------------------------------
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
profit_margin = total_profit / total_sales * 100

loss_orders = (df["Profit"] < 0).sum()
loss_pct = loss_orders / len(df) * 100

print(f"\nTotal Sales: ${total_sales:,.0f}")
print(f"Total Profit: ${total_profit:,.0f}")
print(f"Overall Profit Margin: {profit_margin:.1f}%")
print(f"Loss-making orders: {loss_orders} ({loss_pct:.1f}% of all orders)")

# ------------------------------------------------------------------
# 3. Profitability by Category & Sub-Category
# ------------------------------------------------------------------
category_summary = (
    df.groupby("Category")[["Sales", "Profit"]]
    .sum()
    .assign(Margin=lambda x: x["Profit"] / x["Sales"] * 100)
    .sort_values("Profit", ascending=False)
)
print("\n--- Category Summary ---\n", category_summary)

subcat_summary = (
    df.groupby("Sub-Category")[["Sales", "Profit"]]
    .sum()
    .assign(Margin=lambda x: x["Profit"] / x["Sales"] * 100)
    .sort_values("Profit")
)
print("\n--- Sub-Category Summary (sorted by Profit) ---\n", subcat_summary)

# ------------------------------------------------------------------
# 4. Discount vs Profit
# ------------------------------------------------------------------
bins = [-0.01, 0, 0.2, 0.4, 0.6, 0.8]
labels = ["0%", "0-20%", "20-40%", "40-60%", "60-80%"]
df["Discount Band"] = pd.cut(df["Discount"], bins=bins, labels=labels)

discount_profit = df.groupby("Discount Band")["Profit"].mean()
print("\n--- Avg Profit by Discount Band ---\n", discount_profit)

fig, ax = plt.subplots()
sns.scatterplot(data=df, x="Discount", y="Profit", alpha=0.4, ax=ax)
ax.axhline(0, color="red", linestyle="--", linewidth=1)
ax.set_title("Discount vs Profit (per order)")
plt.tight_layout()
plt.savefig("D:/Superstore/dashboard/discount_vs_profit.png", dpi=150)
plt.close()

# ------------------------------------------------------------------
# 5. Geographic analysis
# ------------------------------------------------------------------
region_summary = (
    df.groupby("Region")[["Sales", "Profit"]]
    .sum()
    .sort_values("Profit", ascending=False)
)
print("\n--- Region Summary ---\n", region_summary)

state_summary = (
    df.groupby("State")[["Sales", "Profit"]]
    .sum()
    .sort_values("Profit")
)
print("\n--- Bottom 5 States by Profit ---\n", state_summary.head())
print("\n--- Top 5 States by Profit ---\n", state_summary.tail())

# ------------------------------------------------------------------
# 6. Segment analysis
# ------------------------------------------------------------------
segment_summary = (
    df.groupby("Segment")[["Sales", "Profit"]]
    .sum()
    .assign(Margin=lambda x: x["Profit"] / x["Sales"] * 100)
)
print("\n--- Segment Summary ---\n", segment_summary)

print("\nEDA complete. Charts saved to D:/Superstore/dashboard/")